In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

In [ ]:
# Taxi-Datensatz von https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
# Data Dictionary: https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf
df = pd.read_parquet('daten/yellow_tripdata_2024-04.parquet')

In [ ]:
# Erster Überblick
df.columns

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.isna().sum()
# Einige Spalten enthalten keine Daten. Betrifft für unsere Fragestellung aber keine relevanten Daten.

In [ ]:
# Korrelation im Überblick
df.corr(numeric_only=True)

In [ ]:
# Alternativ als Diagramm
corr = df.corr(numeric_only=True)
plt.imshow(corr, interpolation="nearest")
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.show()

In [ ]:
df.describe().round(3)
# Hier sind bereits einige Auffälligkeiten erkennbar, wie bspw. unglaublich hohe trip_distance, total_amount etc.

In [ ]:
# Histogramme sind eine gute Wahl, um ein besseres Gefühl für die Daten zu bekommen
df.hist(figsize=(12, 10), bins=50)
plt.tight_layout()
plt.show()

In [ ]:
# Um die Ausreisser bei trip_distance auszufiltern und so ein besseres Histogramm zu bekommen,
# beschränken wir den Wertbereich auf 0 bis 100.
df["trip_distance"][df["trip_distance"] < 100].hist(bins=50)

In [ ]:
(df["trip_distance"] > 100).sum()
# nur 150 (von 3.5M) sind mehr als 100 Meilen gefahren

In [ ]:
(df["trip_distance"] < 0.01).sum()
# Aber immerhin ~46k sind weniger als 0.01 Meilen (~16 Meter) gefahren

In [ ]:
# Wir erzeugen einen neuen Dataframe, bei dem wir solche Werte ausfiltern
filtered_df = df.query("trip_distance > 0.01 and trip_distance < 100")

In [ ]:
filtered_df.shape

In [ ]:
# Histogramm
filtered_df["trip_distance"].hist(bins=50)

In [ ]:
# Scatterplot: Distance vs. Total
# Scatterplots sind ein guter Weg, um schnell den Zusammenhang zwischen zwei Variablen zu sehen.
diag = sns.scatterplot(
    x="trip_distance",
    y="total_amount",
    data=filtered_df,
    alpha=0.2
)
diag.set_title("Distance vs. Total amount")
plt.show()
# Offensichtlich gibt es Ausreisser (hohe Kosten), aber auch unsinnige Werte (negative Kosten)

In [ ]:
df[df["total_amount"] > 1500].head()

In [ ]:
df.sort_values("tolls_amount", ascending=False)[:20]
# Mit Ausnahme des einen Datensatzes ist tolls_amount immer weniger als USD 100

In [ ]:
# Wir filtern alle Datensätze heraus, bei denen die Gesamtkosten weniger als ein Dollar sind
# und bei denen die Mautgebühren größer als 100 Dollar sind.
filtered_df = filtered_df.query("tolls_amount < 100 and total_amount > 1")

In [ ]:
filtered_df.shape

In [ ]:
# Neuer Scatterplot
sns.scatterplot(x=filtered_df["trip_distance"], 
               y=filtered_df["total_amount"],
               alpha=0.1)
# Die frühe Konzentration entlang der Achsen ist immer noch nicht zufriedenstellend,
# aber vorerst kein Problem.

In [ ]:
# Wir verwerfen alle Daten, der ein Fahrtantritt vor dem 1. April war.
filtered_df = filtered_df[filtered_df["tpep_pickup_datetime"] > "2024-04-01 00:00:00"]

In [ ]:
filtered_df.shape

In [ ]:
filtered_df.hist(figsize=(12,14), bins=50)
plt.tight_layout()

In [ ]:
# Als nächstes betreiben wir sogenanntes "Feature Engineering".
# Wir berechnen aus vorhandenen Daten abgeleitete Variablen.
# Hier zum Beispiel im ersten Schritt die "Duration", das heißt die Fahrtdauer.

filtered_df["duration"] = filtered_df["tpep_dropoff_datetime"] - filtered_df["tpep_pickup_datetime"]
filtered_df["duration_min"] = filtered_df["duration"].dt.total_seconds() / 60

In [ ]:
filtered_df["duration_min"].hist(bins=50)
# wäre auch verwunderlich gewesen, wenn es eine Spalte gäbe, die nicht
# einen außergewöhnlichen Maximalwert hätte :o)

In [ ]:
(filtered_df["tpep_dropoff_datetime"] < filtered_df["tpep_pickup_datetime"]).sum()
# Eine Handvoll Datensätze hören auf, bevor sie begonnen haben...

In [ ]:
# Wir entscheiden uns, alle Daten herauszufiltern, die länger als 3 Stunden dauern.
# Diese Grenze ist willkürlich gewählt, erscheint uns aber der Fragestellung angemessen.
filtered_df = filtered_df.query("tpep_dropoff_datetime > tpep_pickup_datetime and duration_min < 180")

In [ ]:
filtered_df["duration_min"].hist(bins=50)
# Nur wenige Fahrten dauern länger als 75 Minuten.

In [ ]:
# Weiteres Feature Engineering:
# Da wir Entfernung und Dauer zur Verfügung haben, rechnen wir uns die durchschnittliche Geschwindigkeit aus.
filtered_df["avg_speed"] = filtered_df["trip_distance"] / (filtered_df['duration_min'] / 60)

In [ ]:
filtered_df["avg_speed"].hist(bins=30)
# So ein Diagramm kann uns nicht mehr überraschen :o)

In [ ]:
# Wir entscheiden uns, alle Durchschnittsgeschwindigkeiten größer als 75mph bzw. weniger als 1mph herauszufiltern.
# Diese Grenzen sind wieder willkürlich gewählt.
filtered_df = filtered_df.query("avg_speed < 75 and avg_speed > 1")

In [ ]:
filtered_df["avg_speed"].hist(bins=30)
# In der Tat ist man in New York im Durchschnitt eher mit 10mph unterwegs.

In [ ]:
filtered_df.shape
# Wir haben inzwischen gut 100.000 Datensätze herausgefiltert. Das entspricht ca. 3% der Datensätze.
# Ob das viel oder wenig ist, ob das relevant ist, oder ob wir bereits eine gefährliche Grenze überschritten haben,
# müssen wir anhand unserer Fragestellung und unserem eigenen Vorwissen (Domänenwissen) beurteilen. 

In [ ]:
df.shape

In [ ]:
# Nochmal ein Blick auf den Scatterplot. Es hat sich nicht viel verändert.
sns.scatterplot(x=filtered_df["trip_distance"], 
               y=filtered_df["total_amount"],
               alpha=0.1)

In [ ]:
filtered_df.describe().round(3)

In [ ]:
# Wir filtern noch zusätzlich alle Datensätze aus, deren Fahrt seit weniger als eine Minute betragen hat.
filtered_df = filtered_df[filtered_df["duration_min"] > 1]

In [ ]:
# stellen wir nun ein erstes Regressionsmodell auf.
model = smf.ols("total_amount ~ trip_distance + duration_min",
               data=filtered_df).fit()

In [ ]:
model.summary()
# Interpretation:
# Intercept ist jener wert bei dem alle X-variablen 0 sind. Bei uns heißt das also, wenn die Entfernung
# (trip_distance) und die Fahrtdauer 0 sind. Dann, so der Koeffizient, sind wir bereits bei 10 USD.
# Anders ausgedrückt, bloßes einsteigen in das Taxi kostet uns laut Modell schon zehn Dollar.
#
# Der Koefficient von trip_distance ist 4, das heißt je gefahrener Meile nehmen die Gesamtkosten um 4 Dollar zu.
# Analog: Jede Minute im Taxi kommen weitere 27 Cent an Gesamtkosten hinzu.
#
# R-squared gibt die erklärte Varianz des Modells an mit 0,87 erklärt unser Modell bereits sehr viel.
# Wichtig: Der Wert alleine ist nicht aussagekräftig genug, um zu definieren, ob das Modell den Datensatz
# gut beschreibt oder sich gut für Vorhersagen eignet.

In [ ]:
# Wir stellen ein zweites Modell auf, bei der wir auch die Höhe des Trinkgeldes mit aufnehmen.
# Hypothese: jeder Dollar zusätzliches Trinkgeld erhöht die Gesamtkosten um eben jenen Dollar.
# (d.h. der erwartete Koeffizient des Trinkgeldes ist 1)
model2 = smf.ols("total_amount ~ trip_distance + duration_min + tip_amount",
               data=filtered_df).fit()

In [ ]:
model2.summary()
# Zu unserem Erstaunen stellen wir fest, dass je Doller Trinkgeld sich die Gesamtsumme um 1.50 Dollar erhöht.
# Das heißt, hier muss noch etwas anderes im Spiel sein.
# Mögliche Erklärungen sind:
# - Es gibt weitere Variablen, die mit der Höhe des Trinkgeldes korrelieren und deren Auswirkung wird,
#   da sie selbst nicht im Modell aufscheinen, im Trinkgeld-Koeffizient subsumiert.
# - Es könnte bspw. eine Korrelation zwischen Fahrtdauer und Trinkgeld geben, die dafür sorgt, dass man 
#   nach längeren Falten mehr Trinkgeld gibt und daher die Gesamtkosten überproportional ansteigen.
# - etc.
#
# Ein Regressionsmodell ist keine Abbildung der Wirklichkeit, sondern es ist ein 
# (hoffentlich brauchbares) Modell. Z.B. könnte es (bei entsprechenden Daten) Szenarien geben,
# wo für jeden Dollar Trinkgeld die Gesamtkosten sogar abnehmen.
# Ein Regressionsmodell erklärt keine kaussalen Zusammenhänge!

In [ ]:
# Rechnen wir uns schnell den prozentmäßigen Anteil des Trinkgeldes aus.
filtered_df["tip_pct"] = filtered_df["tip_amount"] / filtered_df["total_amount"]

In [ ]:
filtered_df["tip_pct"][filtered_df["tip_pct"] < 0.5].hist(bins=50)
# Die Mehrheit gibt ca. 15% Trinkgeld.
# Es gibt aber auch knapp 1 Million Datensätze, wo kein Trinkgeld aufscheint.
# Laut Data Dictionary sind das jene Fälle, wo das Trinkgeld (wahrscheinlich) bar bezahlt worden ist.

In [ ]:
# Schauen wir uns kurz die Residuen, also die Abweichung von tatsächlichem und vorhergesagtem Y-Wert an.
(filtered_df["total_amount"] - model2.fittedvalues).hist(bins=50)
# Offensichtlich gibt es Abweichungen von ca. -600 bis über +200 US Dollar.

In [ ]:
# Residuen als Scatterplot in Abhängigkeit vom verhergesagten Wert
sns.scatterplot(x=model2.fittedvalues, 
               y=filtered_df["total_amount"] - model2.fittedvalues,
               alpha=0.1)
# Der Großteil der Residuen sind einigermaßen gut verteilt.
# Allerdings ist deutlich zu sehen, dass größten Vorhersagefehler bei geringen (vorhergesagten) Kosten entstehen.
# Man kann sich jetzt fragen, ob unser Modell schlecht ist oder ob diese Datensätze (so wie
# viele andere bisher) eigentlich Datenfehler enthalten und ausgefiltert werden sollten.
#
# (Statt Residuen selbst zu berechnen könnten wir auch einfach model2.resid verwenden)

In [ ]:
# Relativer Fehler der Residuen (wird typischerweise nicht verwendet)
sns.scatterplot(x=model2.fittedvalues, 
               y=(filtered_df["total_amount"] - model2.fittedvalues) / model2.fittedvalues,
               alpha=0.1)
# Abweichungen um bis zu dem 70-fachen! Da ist unser Modell schlecht oder (in unserem Fall
# ebenso wahrscheinlich) die Daten nicht in Ordnung.